# Q-factorisation on Gridworld Maze - Combined loss

In [ ]:
from pathlib import Path
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import time
from copy import deepcopy


# Resolve repository src path robustly when running notebook in-place.
repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / 'src').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / 'src'))

from environments.maze_discrete import MazeGridWorld, MazeGoalWrapper
from utils import (TrajectoryReplayBufferDiscrete, evaluate_policy, set_seed, build_goal_batch, get_base_env, collect_valid_states_fourrooms,
                   estimate_fisher_diag, extract_fixed_probe_sa_embedding, extract_mean_sa_embedding, extract_sa_batch_for_isotropy,
                   compute_embedding_drift, collect_weight_snapshot)
from visualisations import visualise_embeddings, visualise_q_table, print_goal_embedding_similarity, plot_full_embedding_dashboard_html
from loss_functions import repulsion_loss_to_memory, sigreg_loss, orthogonal_loss, ewc_regulariser_loss, weight_regulariser_loss
from networks import snapshot_named_parameters, FactorisedDQN_QNetwork
from trainer import dqn_train_multi_loss


DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('Using device:', DEVICE)


In [ ]:
MAZE_LAYOUT = [
    [1,1,1,1,1,1,1,1,1,1,1],
    [1,0,0,0,0,1,0,0,0,0,1],
    [1,0,1,1,0,1,0,1,1,0,1],
    [1,0,1,0,0,0,0,0,1,0,1],
    [1,0,1,0,1,1,1,0,1,0,1],
    [1,0,0,0,1,0,0,0,1,0,1],
    [1,1,1,0,1,0,1,1,1,0,1],
    [1,0,0,0,0,0,1,0,0,0,1],
    [1,0,1,1,1,0,1,1,1,0,1],
    [1,0,0,0,1,0,0,0,0,0,1],
    [1,1,1,1,1,1,1,1,1,1,1],
]

def make_env(goal=(9, 9), slip_prob=0.00, max_horizon=100):
    base = MazeGridWorld(
        maze=MAZE_LAYOUT,
        max_episode_steps=max_horizon,
    )
    env = MazeGoalWrapper(
        base,
        goal_position=goal,
        goal_reward=1.0,
        step_reward=0.0,
        slip_prob=slip_prob,
        reward_mode="simple",
    )
    return env

env = make_env(goal=(9, 9))
obs, info = env.reset()

img = env.unwrapped.render()
goal = env.goal_position

plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.scatter(
    goal[0] * 40 + 20,
    goal[1] * 40 + 20,
    c="lime",
    s=180,
    marker="*",
    edgecolors="black",
)
plt.title(f"Maze with goal at {goal}")
plt.axis("off")
plt.show()

## Training Loop across goals and seeds

In [ ]:
SEEDS = [42, 123, 456, 789, 101112]
GOALS = [(9, 9), (7, 7), (3, 5), (3, 9), (5, 5), (3,6), (9,1), (1,1), (7,4), (9,8),(9,7)]
BUFFER_CAPACITY = 100000
LR = float(1e-3)
sa_keywords_local = ["sa_encoder"]
goal_keywords_local = ["goal_encoder"]

overall_results = {
    goal: {
        "eval_returns": [],
        "min_steps": [],
        "min_time": [],
        "task_embeddings": [],
        "sa_embeddings": [],
        "sa_fixed_probe_embeddings": [],
        "sa_batches_final": [],
    }
    for goal in GOALS
}

for seed in SEEDS:
    print(f"\n================ SEED {seed} ================\n")
    set_seed(seed)

    env_tmp = make_env(goal=GOALS[0])
    obs_dim = env_tmp.observation_space.shape[0]
    num_actions = env_tmp.action_space.n
    env_tmp.close()

    q_net_base = FactorisedDQN_QNetwork(
        obs_dim=obs_dim,
        num_actions=num_actions,
        goal_dim=2,
        hidden_dim=128,
        rep_dim=64,
    ).to(DEVICE)

    q_target_base = FactorisedDQN_QNetwork(
        obs_dim=obs_dim,
        num_actions=num_actions,
        goal_dim=2,
        hidden_dim=128,
        rep_dim=64,
    ).to(DEVICE)

    seed_task_embedding_memory = []
    seen_goal_labels = []
    weight_history = []
    prev_q_net = None
    prev_q_target = None
    cumulative_fisher = None
    prev_reference_params = None

    # replay memories for previous goals
    task_replay_memories = {}

    for goal_idx, goal in enumerate(GOALS):
        print(f"\n----- seed={seed}, goal={goal} -----\n")

        if goal_idx == 0:
            q_net = deepcopy(q_net_base)
            q_target = deepcopy(q_target_base)
            q_target.load_state_dict(q_net.state_dict())

            for p in q_net.parameters():
                p.requires_grad_(True)

            trainable_params = None
            regulariser = None
            fisher_for_this_goal = None
            reference_for_this_goal = None
        else:
            if prev_q_net is None or prev_q_target is None:
                raise ValueError("Previous Q-networks are not available for transfer learning.")

            q_net = deepcopy(prev_q_net)
            q_target = deepcopy(prev_q_target)
            q_target.load_state_dict(q_net.state_dict())

            fisher_for_this_goal = cumulative_fisher
            reference_for_this_goal = prev_reference_params

            for p in q_net.parameters():
                p.requires_grad_(True)

            trainable_params = [p for p in q_net.parameters() if p.requires_grad]
            regulariser = "repulsion"

        weight_history.append(
            collect_weight_snapshot(
                qnet=q_net,
                goal_label=goal,
                stage_label=f"goal_{goal_idx}_before_train_{goal}",
                sa_keywords_local=sa_keywords_local,
                goal_keywords_local=goal_keywords_local,
                max_samples_per_group=40000,
            )
        )

        previous_task_buffers = {
            g: b for g, b in task_replay_memories.items() if g != goal
        }

        (
            q_network,
            q_target_network,
            eval_returns,
            min_steps,
            min_time,
            task_embedding,
            sa_embedding_mean,
            sa_embedding_fixed,
            sa_batch_final,
            buffer,
        ) = dqn_train_multi_loss(
            seed=seed,
            q_network=q_net,
            q_target_network=q_target,
            env=make_env(goal=goal),
            obs_dim=obs_dim,
            buffer_capacity=BUFFER_CAPACITY,
            lr=LR,
            goal=goal,
            device=DEVICE,
            embedding_memory=seed_task_embedding_memory,
            regulariser=None,
            reg_alpha=1,
            params=trainable_params,
            reference_params=reference_for_this_goal,
            fisher_diag=fisher_for_this_goal,
            sa_reg_prefix_filter="sa_encoder",
            make_env=make_env,

            replay_task_buffers=previous_task_buffers,
            replay_ratio=0.25,
            replay_tasks_per_batch=2,
            replay_loss_coef=1.0,
        )

        weight_history.append(
            collect_weight_snapshot(
                qnet=q_network,
                goal_label=goal,
                stage_label=f"goal_{goal_idx}_after_train_{goal}",
                sa_keywords_local=sa_keywords_local,
                goal_keywords_local=goal_keywords_local,   
                max_samples_per_group=40000,
            )
        )

        visualise_q_table(goal, q_network, eval_returns=eval_returns, device=DEVICE, make_env=make_env)
        visualise_embeddings(goal, q_network, device=DEVICE, make_env=make_env)

        seed_task_embedding_memory.append(task_embedding)
        seen_goal_labels.append(str(goal))
        print_goal_embedding_similarity(seed_task_embedding_memory, goal_labels=seen_goal_labels)

        current_reference = snapshot_named_parameters(q_network, prefix_filter="sa_encoder")
        current_fisher = estimate_fisher_diag(
            model=q_network,
            target_model=q_target_network,
            replay_buffer=buffer,
            goal=goal,
            num_actions=num_actions,
            device=DEVICE,
            gamma=0.99,
            batch_size=256,
            n_batches=64,
            prefix_filter="sa_encoder",
        )

        if cumulative_fisher is None:
            cumulative_fisher = current_fisher
        else:
            for name, F_t in current_fisher.items():
                if name in cumulative_fisher:
                    cumulative_fisher[name] = cumulative_fisher[name] + F_t
                else:
                    cumulative_fisher[name] = F_t.clone()

        prev_reference_params = current_reference
        prev_q_net = deepcopy(q_network)
        prev_q_target = deepcopy(q_target_network)

        # store this goal buffer for future replay
        task_replay_memories[goal] = buffer

        overall_results[goal]["eval_returns"].append(eval_returns)
        overall_results[goal]["min_steps"].append(min_steps)
        overall_results[goal]["min_time"].append(min_time)
        overall_results[goal]["task_embeddings"].append(task_embedding)
        overall_results[goal]["sa_embeddings"].append(sa_embedding_mean)
        overall_results[goal]["sa_fixed_probe_embeddings"].append(sa_embedding_fixed)
        overall_results[goal]["sa_batches_final"].append(sa_batch_final)

In [ ]:
goals = list(overall_results.keys())

mean_steps_per_goal = []
std_steps_per_goal = []
mean_time_per_goal = []
std_time_per_goal = []
used_goals = []

for goal in goals:
    steps_raw = overall_results[goal]["min_steps"]
    time_raw  = overall_results[goal]["min_time"]

    steps_arr = np.array(steps_raw).flatten()
    time_arr  = np.array(time_raw).flatten()

    # Skip goals with no data
    if steps_arr.size == 0 or time_arr.size == 0:
        print(f"[WARN] Skipping goal {goal}: empty min_steps or min_time")
        continue

    mean_steps_per_goal.append(steps_arr.mean())
    std_steps_per_goal.append(steps_arr.std())

    mean_time_per_goal.append(time_arr.mean())
    std_time_per_goal.append(time_arr.std())

    used_goals.append(goal)

if not used_goals:
    raise ValueError("No goals with non-empty min_steps/min_time found.")

x = np.arange(len(used_goals))

mean_steps_per_goal = np.array(mean_steps_per_goal)
std_steps_per_goal  = np.array(std_steps_per_goal)
mean_time_per_goal  = np.array(mean_time_per_goal)
std_time_per_goal   = np.array(std_time_per_goal)

fig, (ax1, ax2) = plt.subplots(
    1, 2,
    figsize=(14, 5)
)

# --- Left subplot: steps ---
steps_color = "tab:blue"
ax1.plot(
    x,
    mean_steps_per_goal,
    color=steps_color,
    marker="o",
    linewidth=2.5,
    label="Mean Min Steps"
)
ax1.fill_between(
    x,
    mean_steps_per_goal - std_steps_per_goal,
    mean_steps_per_goal + std_steps_per_goal,
    color=steps_color,
    alpha=0.2,
    label="±1 std"
)
ax1.set_title(F"Minimum Steps Across {len(SEEDS)} Seeds")
ax1.set_xlabel("Goal Index")
ax1.set_ylabel("Steps")
ax1.set_xticks(x)
ax1.set_xticklabels([str(g) for g in used_goals], rotation=45, ha="right")
ax1.grid(True, axis="y", linestyle="--", alpha=0.3)
ax1.legend(loc="best")

# --- Right subplot: time ---
time_color = "tab:orange"
ax2.plot(
    x,
    mean_time_per_goal,
    color=time_color,
    marker="o",
    linewidth=2.5,
    label="Mean Min Time"
)
ax2.fill_between(
    x,
    mean_time_per_goal - std_time_per_goal,
    mean_time_per_goal + std_time_per_goal,
    color=time_color,
    alpha=0.2,
    label="±1 std"
)
ax2.set_title(F"Minimum Time Across {len(SEEDS)} Seeds")
ax2.set_xlabel("Goal Index")
ax2.set_ylabel("Time")
ax2.set_xticks(x)
ax2.set_xticklabels([str(g) for g in used_goals], rotation=45, ha="right")
ax2.grid(True, axis="y", linestyle="--", alpha=0.3)
ax2.legend(loc="best")

fig.suptitle("Mean ± Std per Goal", fontsize=14)
fig.tight_layout()
plt.show()

## Backward Compatibility test (or new goal test)

Do not retrain anything, just give the task embedding, and check whether the sa encoder can give good Q functions still. The task embedding must be previously learned.

In [ ]:
GOALS = [(9, 9), (9,8), (9,7), (5,7)]

drift = compute_embedding_drift(overall_results, q_network, DEVICE)
for goal, cos in drift.items():
    print(goal, "cos(old, final) =", cos)

for goal in GOALS:

    if goal not in overall_results:
        task_embedding = q_network.encode_goal(torch.tensor(goal, dtype=torch.float32, device=DEVICE).unsqueeze(0)).detach().cpu().numpy()
    else:
        task_embedding = overall_results[goal]["task_embeddings"]
    visualise_q_table(goal, q_network, eval_returns=None, task_embedding=task_embedding, device=DEVICE, make_env=make_env)
    visualise_embeddings(goal, q_network, device=DEVICE, make_env=make_env)

## HTML visualisation

In [ ]:
for goal, data in overall_results.items():
    print(goal)
    print("task_embeddings:", len(data.get("task_embeddings", [])))
    print("sa_fixed_probe_embeddings:", len(data.get("sa_fixed_probe_embeddings", [])))
    print("sa_batches_final:", len(data.get("sa_batches_final", [])))

dashboard = plot_full_embedding_dashboard_html(
    overall_results=overall_results,
    qnet=q_network,
    mode="all",
    save_html="plots/full_embedding_dashboard_maze.html",
    weights_his=weight_history,
)

print(dashboard["save_html"])
print(dashboard["isotropy_metrics"])